Importar librerías necesarias.

In [118]:
import pandas as pd
import numpy as np
from datetime import datetime as dt

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import classification_report, accuracy_score

import joblib

import os

root_data_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))

os.environ["KAGGLEHUB_CACHE"] = root_data_dir

import kagglehub

path = kagglehub.dataset_download("martj42/international-football-results-from-1872-to-2017")

print("Dataset descargado correctamente en:", path)

path = kagglehub.dataset_download("martj42/international-football-results-from-1872-to-2017")

results = pd.read_csv("../data/datasets/martj42/international-football-results-from-1872-to-2017/versions/128/results.csv")

Dataset descargado correctamente en: c:\Users\fede_\Documents\Proyectos Fede\Prode\data\datasets\martj42\international-football-results-from-1872-to-2017\versions\128


Función para actualizar dataset, de ser necesario.

In [119]:
def update_matches(team1, team2, score1, score2, date, tournament, neutral, df):
    df.loc[
    (df['date'] == date) &
    (df['tournament'] == tournament) &
    (df['neutral'] == neutral) & 
    (df['home_team'] == team1) & 
    (df['away_team'] == team2), 
    ['home_score', 'away_score']
] = [score1, score2]
    
    
    return df

update_matches('South Africa', 'Canada', 0, 1, '2026-06-28', 'FIFA World Cup', True, results)

results.sort_values(by=['date'], ascending=False).head(30)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49492,2026-07-03,Colombia,Ghana,NaN,NaN,FIFA World Cup,Kansas City,United States,True
49491,2026-07-03,Argentina,Cape Verde,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49490,2026-07-03,Australia,Egypt,NaN,NaN,FIFA World Cup,Arlington,United States,True
49489,2026-07-02,Switzerland,Algeria,NaN,NaN,FIFA World Cup,Vancouver,Canada,True
49488,2026-07-02,Portugal,Croatia,NaN,NaN,FIFA World Cup,Toronto,Canada,True
49487,2026-07-02,Spain,Austria,NaN,NaN,FIFA World Cup,Inglewood,United States,True
49486,2026-07-01,United States,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Santa Clara,United States,False
49485,2026-07-01,Belgium,Senegal,NaN,NaN,FIFA World Cup,Seattle,United States,True
49484,2026-07-01,England,DR Congo,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49482,2026-06-30,France,Sweden,NaN,NaN,FIFA World Cup,East Rutherford,United States,True


Comenzar proceso de limpieza de datos.

In [120]:
results.count()

date          49493
home_team     49493
away_team     49493
home_score    49478
away_score    49478
tournament    49493
city          49493
country       49493
neutral       49493
dtype: int64

In [121]:
results.dtypes

date           object
home_team      object
away_team      object
home_score    float64
away_score    float64
tournament     object
city           object
country        object
neutral          bool
dtype: object

In [122]:
results_modern = results[(results["date"] >= "2000-01-01") & (results["date"] <= "2026-12-31")].copy() # Crear sub-dataset consistente de los encuentros del año 2000 en adelante, que se usara para entrenar a los modelos.
results_modern = results_modern.drop(['city', 'country'], axis=1)                                      # Dropear columnas que no aportan valor.

In [123]:
results_modern.sort_values(by=['date'], ascending=False).head(30)

,date,home_team,away_team,home_score,away_score,tournament,neutral
49492,2026-07-03,Colombia,Ghana,NaN,NaN,FIFA World Cup,True
49491,2026-07-03,Argentina,Cape Verde,NaN,NaN,FIFA World Cup,True
49490,2026-07-03,Australia,Egypt,NaN,NaN,FIFA World Cup,True
49489,2026-07-02,Switzerland,Algeria,NaN,NaN,FIFA World Cup,True
49488,2026-07-02,Portugal,Croatia,NaN,NaN,FIFA World Cup,True
49487,2026-07-02,Spain,Austria,NaN,NaN,FIFA World Cup,True
49486,2026-07-01,United States,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,False
49485,2026-07-01,Belgium,Senegal,NaN,NaN,FIFA World Cup,True
49484,2026-07-01,England,DR Congo,NaN,NaN,FIFA World Cup,True
49482,2026-06-30,France,Sweden,NaN,NaN,FIFA World Cup,True


Casteo y limpieza de datos faltantes.

In [124]:
results_modern = results_modern.dropna()  # Dropeo de columnas nulas (el dataset ya está limpio, son únicamente los partidos que aún no se disputaron.)

In [125]:
results_modern[['home_score', 'away_score']] = results_modern[['home_score', 'away_score']].astype(int)  # Casteo de variables.
results_modern['date'] = pd.to_datetime(results_modern['date'])

## Feature Engineering
Crear variables de promedio de goles a favor y en contra para ambos equipos, en los últimos 5 pártidos, en la fecha de disputa del partido cuyo resultado se busca predecir.

In [126]:
results_modern = results_modern.sort_values("date").reset_index(drop=True)

def get_team_history(df):
    home = df[['date', 'home_team', 'home_score', 'away_score']].rename(                            # Función para obtener historial de equipos disputados de forma separada.
        columns={'home_team': 'team', 'home_score': 'goals_for', 'away_score': 'goals_against'}
    )
    away = df[['date', 'away_team', 'away_score', 'home_score']].rename(
        columns={'away_team': 'team', 'away_score': 'goals_for', 'home_score': 'goals_against'}
    )
    return pd.concat([home, away]).sort_values(['team', 'date'])

team_history = get_team_history(results_modern)

team_history['roll_gf_5'] = team_history.groupby('team')['goals_for'].transform(lambda x: x.shift(1).rolling(5).mean())         # Calcula el promedio de goles a favor y en contra en los últimos 5 partidos sin contar 
team_history['roll_ga_5'] = team_history.groupby('team')['goals_against'].transform(lambda x: x.shift(1).rolling(5).mean())     # el actual. La rolling window agrupa filas de a 5 a la vez.  

results_modern = results_modern.merge(
    team_history[['date', 'team', 'roll_gf_5', 'roll_ga_5']],                                                                   # Mergear con el dataframe principal. 
    left_on=['date', 'home_team'], right_on=['date', 'team'], how='left'
).rename(columns={'roll_gf_5': 'home_goals_for_5', 'roll_ga_5': 'home_goals_against_5'}).drop(columns=['team'])

results_modern = results_modern.merge(
    team_history[['date', 'team', 'roll_gf_5', 'roll_ga_5']],
    left_on=['date', 'away_team'], right_on=['date', 'team'], how='left'
).rename(columns={'roll_gf_5': 'away_goals_for_5', 'roll_ga_5': 'away_goals_against_5'}).drop(columns=['team'])


Crear variable de resultado (0 para empate, 1 para victoria local, 2 para victoria visitante)

In [127]:
results_modern.loc[results_modern["home_score"] > results_modern["away_score"], "result"] = 1        
results_modern.loc[results_modern["home_score"] < results_modern["away_score"], "result"] = 2
results_modern.loc[results_modern["home_score"] == results_modern["away_score"], "result"] = 0

## Establecer pesos
Los encuentros tienen distinto peso de acuerdo a su contexto. Si ocurren en una copa mundial, tienen más peso, en otros torneos, menos peso, y si son amistosos, incluso menos.
Además, los encuentros pierden peso con el paso del tiempo, de manera que un encuentro reciente tiene mucho más peso en la predicción que uno de hace varios años.

In [128]:
def calculate_match_weight(match_date, tournament_type):

    years_old = (dt.today() - match_date).days / 365
    time_weight = np.exp(-0.138 * years_old)

    if tournament_type == "FIFA World Cup":
        tournament_weight = 1.0
    elif tournament_type == "Friendly":
        tournament_weight = 0.3
    else:
        tournament_weight = 0.7

    return time_weight * tournament_weight

weights = results_modern.apply(
    lambda row: calculate_match_weight(row["date"], row["tournament"]),
    axis=1
)

results_modern.sort_values(by=['date'], ascending=False).head(30)


,date,home_team,away_team,home_score,away_score,tournament,neutral,home_goals_for_5,home_goals_against_5,away_goals_for_5,away_goals_against_5,result
25437,2026-06-28,South Africa,Canada,0,1,FIFA World Cup,True,0.6,0.6,2.2,0.8,2.0
25436,2026-06-27,Colombia,Portugal,0,0,FIFA World Cup,True,2.0,1.0,2.4,0.6,0.0
25435,2026-06-27,Algeria,Austria,3,3,FIFA World Cup,True,1.4,0.8,2.0,0.8,0.0
25434,2026-06-27,Jordan,Argentina,1,3,FIFA World Cup,True,1.0,2.6,3.0,0.0,2.0
25433,2026-06-27,Croatia,Ghana,2,1,FIFA World Cup,True,1.2,2.0,0.6,1.0,1.0
25432,2026-06-27,DR Congo,Uzbekistan,3,1,FIFA World Cup,True,0.6,0.8,0.4,2.4,1.0
25431,2026-06-27,Panama,England,0,2,FIFA World Cup,True,1.4,2.2,1.6,0.6,2.0
25428,2026-06-26,Egypt,Iran,1,1,FIFA World Cup,True,1.2,0.8,2.4,0.6,0.0
25425,2026-06-26,Uruguay,Spain,0,1,FIFA World Cup,True,1.0,1.8,1.6,0.4,2.0
25426,2026-06-26,Cape Verde,Saudi Arabia,0,0,FIFA World Cup,True,1.2,1.2,1.0,1.4,0.0


Encoding de variables categóricas

In [129]:
team_encoder = LabelEncoder()

all_teams = pd.concat([results_modern['home_team'], results_modern['away_team']])
team_encoder.fit(all_teams)

joblib.dump(team_encoder, "../app/team_encoder.pkl")

results_modern['home_team_encoded'] = team_encoder.transform(results_modern['home_team'])
results_modern['away_team_encoded'] = team_encoder.transform(results_modern['away_team'])


tournament_encoder = LabelEncoder()
results_modern['tournament_encoded'] = tournament_encoder.fit_transform(results_modern['tournament'])

joblib.dump(tournament_encoder, "../app/tournament_encoder.pkl")

results_modern['neutral'] = results_modern['neutral'].astype(int)


results_proccesed = results_modern[[
    'date', 
    'home_team', 'away_team', 'tournament',
    'home_team_encoded', 'away_team_encoded', 
    'home_goals_for_5', 'home_goals_against_5', 
    'away_goals_for_5', 'away_goals_against_5', 
    'tournament_encoded', 'neutral'
]].copy()

columns_float = ['home_goals_for_5', 'home_goals_against_5', 'away_goals_for_5', 'away_goals_against_5']
results_proccesed[columns_float] = results_proccesed[columns_float].astype(float)

results_proccesed.to_csv("../app/results_proccesed.csv", index=False)

Split de train/test. No se usó un train_test_split común para evitar que el modelo trate de predecir partidos del pasado con datos del futuro. Para train se usan datos hasta el 2024, y para test datos del 2024 hasta el presente.

In [130]:
cutoff_date = pd.to_datetime("2024-01-01")


train_df = results_modern[results_modern["date"] < cutoff_date]
test_df = results_modern[results_modern["date"] >= cutoff_date]

features = ['home_team_encoded', 'away_team_encoded', 'tournament_encoded', 'neutral', 'home_goals_for_5', 'home_goals_against_5', 'away_goals_for_5', 'away_goals_against_5']

X_train = train_df[features]
y_train_home = train_df['home_score']
y_train_away = train_df['away_score']

X_test = test_df[features]
y_test_home = test_df['home_score']
y_test_away = test_df['away_score']

In [131]:
model_home = RandomForestRegressor(n_estimators=100, random_state=1905) # 19/05 es mi cumpleaños :)

model_away = RandomForestRegressor(n_estimators=100, random_state=1905)

X_train_weights = weights.loc[X_train.index]


model_home.fit(X_train, y_train_home, sample_weight=X_train_weights)
model_away.fit(X_train, y_train_away, sample_weight=X_train_weights)


RandomForestRegressor(random_state=1905)

Predicción y redondeo.

In [132]:

pred_home_continuous = model_home.predict(X_test)
pred_away_continuous = model_away.predict(X_test)


pred_home_rounded = np.round(pred_home_continuous).astype(int)
pred_away_rounded = np.round(pred_away_continuous).astype(int)

Evaluación del rendimiento mediante MAE.

In [133]:
mae_home = mean_absolute_error(y_test_home, pred_home_continuous)
mae_away = mean_absolute_error(y_test_away, pred_away_continuous)

print(print(f"MAE Goles Local: {mae_home:.2f} goles de diferencia promedio."))
print(f"MAE Goles Visitante: {mae_away:.2f} goles de diferencia promedio.")

MAE Goles Local: 1.17 goles de diferencia promedio.
None
MAE Goles Visitante: 0.94 goles de diferencia promedio.


Más métricas de rendimiento.

In [134]:
actual_result = test_df['result']                                           # Reconstruir el resultado real en el set de test.

pred_result = np.where(pred_home_rounded > pred_away_rounded, 1,            # Reconstruir el resultado predicho a partir de los goles redondeados.
np.where(pred_home_rounded < pred_away_rounded, 2, 0))

prode_accuracy = accuracy_score(actual_result, pred_result)                 # Calcular el accuracy del Prode.
print(f"Precisión del Prode: {prode_accuracy * 100:.2f}%")

print("\nReporte de Clasificación del Prode:")                              # Ver el desglose por categoría.
print(classification_report(actual_result, pred_result, target_names=['Empate (0)', 'Gana Local (1)', 'Gana Visitante (2)']))

Precisión del Prode: 47.81%

Reporte de Clasificación del Prode:
                    precision    recall  f1-score   support

        Empate (0)       0.27      0.39      0.32       623
    Gana Local (1)       0.60      0.64      0.62      1249
Gana Visitante (2)       0.54      0.28      0.37       757

          accuracy                           0.48      2629
         macro avg       0.47      0.44      0.44      2629
      weighted avg       0.51      0.48      0.48      2629



Función para obtener datos de dos equipos para hacer una prediccoón.

In [135]:
def get_match_features_live(home_team, away_team, tournament_name, is_neutral, results_df, team_enc, tourn_enc, feature_columns):
    today = pd.to_datetime(dt.today().date())
    
    home_history = results_df[                                                                  # Filtrar el historial histórico general de cada equipo antes de hoy
        ((results_df['home_team'] == home_team) | (results_df['away_team'] == home_team)) & 
        (pd.to_datetime(results_df['date']) < today)
    ].sort_values('date').tail(5)
    
    away_history = results_df[
        ((results_df['home_team'] == away_team) | (results_df['away_team'] == away_team)) & 
        (pd.to_datetime(results_df['date']) < today)
    ].sort_values('date').tail(5)
    

    if len(home_history) < 5 or len(away_history) < 5:                                          # Checkear si existe historial suficiente para evaluar.
        raise ValueError("Uno de los equipos no tiene al menos 5 partidos previos en el dataset para calcular las métricas.")
        
   
    home_gf = []
    home_gc = []
    for _, row in home_history.iterrows():      # Calcular goles a favor y en contra de local.
        if row['home_team'] == home_team:
            home_gf.append(row['home_score'])
            home_gc.append(row['away_score'])
        else:
            home_gf.append(row['away_score'])
            home_gc.append(row['home_score'])
            
    away_gf = []
    away_gc = []                                # Calcular goles a favor y en contra de visitante.
    for _, row in away_history.iterrows():
        if row['home_team'] == away_team:
            away_gf.append(row['home_score'])
            away_gc.append(row['away_score'])
        else:
            away_gf.append(row['away_score'])
            away_gc.append(row['home_score'])
            
    home_encoded = team_enc.transform([home_team])[0]               # Codificar variables categóricas.
    away_encoded = team_enc.transform([away_team])[0]
    tournament_encoded = tourn_enc.transform([tournament_name])[0]
    
    live_features = pd.DataFrame([{                                 # 5. Estructurar el dataframe producto.
        'neutral': int(is_neutral),
        'home_goals_for_5': np.mean(home_gf),
        'home_goals_against_5': np.mean(home_gc),
        'away_goals_for_5': np.mean(away_gf),
        'away_goals_against_5': np.mean(away_gc),
        'home_team_encoded': home_encoded,
        'away_team_encoded': away_encoded,
        'tournament_encoded': tournament_encoded
    }])
    
    return live_features[feature_columns]


input_data = get_match_features_live(
    home_team='Netherlands',
    away_team='Tunisia',
    tournament_name='FIFA World Cup',
    is_neutral=1,
    results_df=results_modern,
    team_enc=team_encoder,
    tourn_enc=tournament_encoder,
    feature_columns=features
)

pred_home = model_home.predict(input_data)[0]
pred_away = model_away.predict(input_data)[0]

print(f"Predicción exacta: Netherlands {pred_home:.2f} - {pred_away:.2f} Tunisia")
print(f"PRODE: Netherlands {int(np.round(pred_home))} - {int(np.round(pred_away))} Tunisia")

Predicción exacta: Netherlands 4.87 - 0.66 Tunisia
PRODE: Netherlands 5 - 1 Tunisia


Función para predecir resultados en terreno neutral. (Perspectiva local-visitante, visitante-local, calculando el promedio entre ellas)

In [139]:
def predict_neutral_match_symmetric(team_A, team_B, tournament_name, results_df, team_enc, tourn_enc, feature_columns, model_home, model_away):
    
    features_1 = get_match_features_live(
        home_team=team_A, away_team=team_B, tournament_name=tournament_name,
        is_neutral=1, results_df=results_df, team_enc=team_enc, tourn_enc=tourn_enc,
        feature_columns=feature_columns
    )
    pred_home_1 = model_home.predict(features_1)[0]
    pred_away_1 = model_away.predict(features_1)[0]
    
    features_2 = get_match_features_live(
        home_team=team_B, away_team=team_A, tournament_name=tournament_name,
        is_neutral=1, results_df=results_df, team_enc=team_enc, tourn_enc=tourn_enc,
        feature_columns=feature_columns
    )
    
    pred_home_2 = model_home.predict(features_2)[0]
    pred_away_2 = model_away.predict(features_2)[0]
    
    final_goles_team_A = (pred_home_1 + pred_away_2) / 2
    final_goles_team_B = (pred_away_1 + pred_home_2) / 2
    
    prode_A = int(np.round(final_goles_team_A))
    prode_B = int(np.round(final_goles_team_B))
    
    print(f"Predicción Continua: {team_A} {final_goles_team_A:.2f} - {final_goles_team_B:.2f} {team_B}")
    print(f"Marcador PRODE: {team_A} {prode_A} - {prode_B} {team_B}")
    
    return prode_A, prode_B

predict_neutral_match_symmetric(
    team_A='Canada',
    team_B='Saudi Arabia',
    tournament_name='FIFA World Cup',
    results_df=results_modern,
    team_enc=team_encoder,
    tourn_enc=tournament_encoder,
    feature_columns=features,
    model_home=model_home,
    model_away=model_away
)

Predicción Continua: Canada 1.79 - 0.89 Saudi Arabia
Marcador PRODE: Canada 2 - 1 Saudi Arabia


(2, 1)

In [137]:
features = ['home_team_encoded', 'away_team_encoded', 'tournament_encoded', 'neutral', 'home_goals_for_5', 'home_goals_against_5', 'away_goals_for_5', 'away_goals_against_5']

X_train = results_modern[features]
y_train_home = results_modern['home_score']
y_train_away = results_modern['away_score']

model_home = RandomForestRegressor(n_estimators=100, random_state=1905) 

model_away = RandomForestRegressor(n_estimators=100, random_state=1905)

X_train_weights = weights.loc[X_train.index]


model_home.fit(X_train, y_train_home, sample_weight=X_train_weights)
model_away.fit(X_train, y_train_away, sample_weight=X_train_weights)

joblib.dump(model_home, "../app/model_home.pkl", compress=3)
joblib.dump(model_away, "../app/model_away.pkl", compress=3)

['../app/model_away.pkl']